# 05 · Per-Benchmark Breakdown

**Purpose:** show the **mean value per (benchmark × language)** as bar charts — one
figure per (benchmark × metric), 18 language bars each, sorted most-efficient first
and coloured by execution compiler. Complements the aggregate rankings in
notebooks 02–04 by exposing per-benchmark behaviour.

**24 figures** = 8 CLBG benchmarks × 3 metrics:
1. CPU Energy (J)
2. Memory / DRAM Energy (J)
3. Execution Time (s)

Each is written to `notebooks/figures/per_benchmark/<benchmark>_<metric>.pdf` as a
300-dpi vector PDF.

Values are the per-(language × benchmark) cell means from `results_clean_runs.csv`
(`plot_style.cell_means`). Each figure has its own x-scale, so bars show the ranking
**within** that benchmark.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # make the shared style module importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from itertools import combinations

import importlib
import plot_style as ps
importlib.reload(ps)   # pick up edits to plot_style.py without a kernel restart
ps.apply_style()

# Canonical constants — single source: plot_style.
COL_CPU_ENERGY, COL_MEM_ENERGY = ps.COL_CPU_ENERGY, ps.COL_MEM_ENERGY
COL_TIME                       = ps.COL_TIME
COMPILER        = ps.COMPILER
COMPILER_COLORS = ps.COMPILER_COLORS
COMPILER_ORDER  = ps.COMPILER_ORDER

# Single source of truth: per-run rows (df) + per-cell means with EDP (df_mean).
df      = ps.load_runs()
df_mean = ps.cell_means(df)
print(f"Cell-means: {df_mean.shape} | "
      f"{df['language'].nunique()} languages \u00d7 {df['benchmark'].nunique()} benchmarks")

In [ ]:
SUBDIR = 'per_benchmark'   # figures/per_benchmark/<benchmark>_<metric>.pdf


def per_benchmark_figure(benchmark, metric_col, axis_label, metric_slug):
    """Horizontal bar chart of one benchmark's 18 language means for one metric.

    Bars are sorted ascending (most efficient at the top), coloured by execution
    compiler, and labelled with their value. Inputs: the benchmark name, the metric
    column, an axis label (with unit), and a short metric slug used in the filename.
    Reads df_mean. Side effect: writes figures/per_benchmark/<benchmark>_<slug>.pdf
    at 300 dpi. Returns the output Path.
    """
    sub = (df_mean[df_mean['benchmark'] == benchmark]
           .set_index('language')[metric_col]
           .sort_values())

    fig, ax = plt.subplots(figsize=(8, 6))
    colors = [ps.compiler_color(l) for l in sub.index]
    bars = ax.barh(sub.index, sub.values, color=colors, alpha=0.85, edgecolor='white')

    x_max = sub.max()
    for bar, val in zip(bars, sub.values):
        ax.text(bar.get_width() + x_max * 0.012, bar.get_y() + bar.get_height() / 2,
                f'{val:,.2f}', va='center', ha='left', fontsize=8, color='#333333')

    ax.set_xlim(0, x_max * 1.20)
    ax.invert_yaxis()                 # most efficient (lowest) at the top
    ax.set_title(benchmark, fontsize=13)
    ax.set_xlabel(axis_label, fontsize=10)
    ax.tick_params(labelsize=9)
    ax.grid(False)
    ax.grid(axis='x', linestyle='--', alpha=0.4)

    legend_handles = [mpatches.Patch(color=COMPILER_COLORS[p], label=p, alpha=0.85)
                      for p in COMPILER_ORDER]
    ax.legend(handles=legend_handles, title='Execution Model',
              loc='upper right', fontsize=8, title_fontsize=9, framealpha=0.9)

    plt.tight_layout()
    out = ps.save_fig(fig, f'{SUBDIR}/{benchmark}_{metric_slug}')
    plt.show()
    return out


def export_all_benchmarks(metric_col, axis_label, metric_slug):
    """Render one figure per benchmark for a single metric.

    Inputs: the metric column, its axis label, and the filename slug. Iterates the
    benchmarks in df_mean in sorted order, calling per_benchmark_figure for each.
    Side effect: writes one PDF per benchmark and prints the count. Returns the list
    of output Paths.
    """
    benchmarks = sorted(df_mean['benchmark'].unique())
    paths = [per_benchmark_figure(bm, metric_col, axis_label, metric_slug)
             for bm in benchmarks]
    print(f'{len(paths)} figures written to figures/{SUBDIR}/ ({metric_slug})')
    return paths

## 1. CPU Energy per Benchmark (J)

In [ ]:
export_all_benchmarks(COL_CPU_ENERGY, 'CPU Energy (J)', 'cpu_energy')

> **Takeaway:** AOT native compilers lead on every benchmark; the interpreted languages' energy penalty is widest on the compute-bound benchmarks (n-body, spectral-norm, mandelbrot, fannkuch-redux).

## 2. Memory Energy per Benchmark (J)

In [ ]:
export_all_benchmarks(COL_MEM_ENERGY, 'Memory Energy (J)', 'mem_energy')

> **Takeaway:** DRAM energy is low and fairly uniform on most benchmarks; regex-redux is the outlier, where Erlang's very long run dominates memory energy.

## 3. Execution Time per Benchmark (s)

In [ ]:
export_all_benchmarks(COL_TIME, 'Execution Time (s)', 'time')

> **Takeaway:** the speed ordering is stable across benchmarks (AOT fastest, interpreted slowest); regex-redux and k-nucleotide are the most time-expensive workloads overall.